# Milestone - Unit 10: model predictive control

Unit 10 asked whether steering that plans the whole next 2 s, every tick, can race where pure pursuit, which looks at one point, cannot.
`agents/unit-10.py` is Unit 9's file (the racing line, the measured grip, the speed profile and its PI loop) with `STEERING = "mpc"`: every tick it plans `MPC_HORIZON` = 40 ticks of path curvature by linear time-varying least squares on the path-frame error model, applies the first command, and plans again on the next tick.
The model knows the measured lag of 1 tick (Unit 8) and the measured steering gain (66 deg); it is a **kinematic** model, and the car it races is not.
It plans with `MU_FRACTION = 0.58` of Unit 8's measured grip, re-tuned for MPC on the training family by the margin rule.

The notebook re-derives the model's stacked solve as student code, checks it against the shipped functions, and asserts the unit's gates (plan `roar-009`) on the trimmed subset below; a routed racekit test (`tests/test_racekit_unit10_premise.py`) races every multi-seed figure in full:

- **U10.1** the solve (race-free): G and h built one tick at a time equal the closed form (and the shipped `stack_horizon`) to 1e-12; on the tightest corner of `test-circuit`'s line the least-squares plan, evaluated on the nonlinear model, costs 3.53258 against `scipy.optimize.minimize`'s 3.49971 (0.94% above, within 2%), and the two plans differ by at most 1e-3 1/m;
- **U10.2** the same fraction: at 0.54 on `test-circuit`, seeds 0-4, MPC is clean (max line error at most 1.3 m; seed 0 279.65 s) and **neither** MPC nor Unit 9's pursuit (279.60 s) is noise-aware better than the other;
- **U10.3** the hook: at 0.58 on `test-circuit-seed4`, seed 0, pursuit (`agents/unit-09.py`) is not clean and MPC is clean in 271.10 s;
- **U10.4** the edge: at 0.60, MPC is clean on `test-circuit` in 272.40 s (seed 0); at 0.62 on `test-circuit-seed1` it is not clean (seed 0);
- **U10.5** the transfer: on the unseen `test-circuit-seed2026`, seed 0, the shipped 0.58 is clean in 276.85 s, faster than Unit 9's 0.54 (282.05 s, the median of five, pinned in `roar-008`);
- **U10.6** the lag: with `MPC_LAG = 0`, at 0.60 on `test-circuit`, MPC is not clean (seed 0);
- **U10.7** the tick budget: with `MPC_BUDGET_MS = 0.0` (every `mpc()` call over budget) the horizon is 40, 20, then 10 for the rest of the race, `over_budget` counts every call, and the race on `test-circuit-seed2026` is still clean (277.30 s, seed 0); a non-finite plan applies the shifted last finite plan, counted in `fallbacks`, and pure pursuit is never called; with `STEERING = "pursuit"` the file races exactly like `agents/unit-09.py`; `check_submission` passes;
- **U10.8** `monza-like` at 0.64, seed 0: MPC is clean in 468.75 s; pursuit finishes but is not clean, because its max line error is just over the 2 m rule (a tracking-margin result, not a crash).

Every race uses racekit's carla-like car, `carla_like(noise=True)`, seeded by `run_race`, with `max_seconds` 450 on the `test-circuit` family and 900 on `monza-like`.
Every pinned MPC race runs the variant `MPC_BUDGET_MS = None`: the shipped budget reads a wall clock, so on a busy machine it could shorten the horizon at a different tick, and the race would depend on the machine (U10.7 races the two deterministic extremes).
Tolerances (the plan's Gates): times within two ticks (±0.10 s) of the pinned value; outcomes exact; "not clean" is asserted, never a collision count.
A race is **clean** when it finished, with 0 collisions, and its max |cross-track error from its own line| (the centreline offset minus n at the car's waypoint (progress + 1) mod n, over the driving rows) is at most 2 m.
The per-track constants (`MU_G`, Unit 8's measured grip, and `LINE_CONTROL`, Unit 9's line) are written in with `write_variant`.
Every figure is a model number on racekit's carla-like car, not CARLA's.

## 1. Set-up

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.
The line solver is Unit 9's (QR, then bounded least squares); the model's stacked solve is Unit 10's lesson code, written out again here.

In [ ]:
import math
import sys
import time
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear, minimize

AGENTS = Path("..") / "agents"
UNIT_09, UNIT_10 = AGENTS / "unit-09.py", AGENTS / "unit-10.py"
SEEDS = (0, 1, 2, 3, 4)
TICKS_2 = 0.10 + 1e-9  # s: two ticks, the plan's tolerance
CLEAN_ERROR = 2.0  # m
STEP, MARGIN, PIN = 10, 2.5, 30
NO_TIMING = {"MPC_BUDGET_MS": None}  # every pinned MPC race: no wall clock in the race
MEASURED_MU_G = {  # Unit 8's grip probes, mu * g per section (m/s^2, rounded to 0.01)
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
    "test-circuit-seed2026": (7.70, 7.69, 7.70, 9.81, 8.16, 8.04),
    "monza-like": (5.38, 8.96, 9.12, 9.14, 9.70, 7.97),
}
workdir = Path(mkdtemp(prefix="roar-milestone-10-"))


# --- Unit 9: the racing line ---------------------------------------------------------------------
def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def spline_basis(n, step=STEP):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track):
    # The smoothest line's control values (QR + bvls), as Unit 9 solves it.
    x, y = track.x, track.y
    n = len(x)
    nx, ny = normals(x, y)
    basis = spline_basis(n)
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    index = np.arange(n)
    fixed = (index < PIN) | (index >= n - PIN)
    high = np.zeros(basis.shape[1])
    for j in range(basis.shape[1]):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(fixed[support]) > 0 else np.min(track.half_width[support] - MARGIN)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


# --- Unit 10: the model, stacked two ways, and the least-squares plan ------------------------------
def stack_by_ticks(e, psi, a, kappa):
    # G and h one tick at a time: x_{t+1} = A_t x_t + b_t u_t + c_t, with rows e_1, psi_1, e_2, psi_2, ...
    m = len(a)
    G = np.zeros((2 * m, m))
    h = np.zeros(2 * m)
    state = np.array([e, psi])
    effect = np.zeros((2, m))  # column j: what command j has done to the current state so far
    for t in range(m):
        A = np.array([[1.0, a[t]], [0.0, 1.0]])
        state = A @ state + np.array([0.0, -a[t] * kappa[t]])
        effect = A @ effect
        effect[:, t] = (0.0, a[t])
        G[2 * t:2 * t + 2] = effect
        h[2 * t:2 * t + 2] = state
    return G, h


def stack_horizon(e, psi, a, kappa):
    # The closed form: G_psi[t, j] = a_j and G_e[t, j] = a_j (S_t - S_j) for j <= t, S the running sum of a.
    m = len(a)
    S = np.cumsum(a)
    G_psi = np.tril(np.tile(a, (m, 1)))
    G_e = np.tril(S[:, None] - S[None, :]) * a[None, :]
    G = np.empty((2 * m, m))
    h = np.empty(2 * m)
    G[0::2], G[1::2] = G_e, G_psi
    h[0::2], h[1::2] = e + S * psi - G_e @ kappa, psi - G_psi @ kappa
    return G, h


def plan_least_squares(G, h, u_last, q_e, q_psi, r):
    # min sum q_e e^2 + q_psi psi^2 + r sum (u_t - u_{t-1})^2 as one stacked least-squares problem
    m = G.shape[1]
    w = np.tile([math.sqrt(q_e), math.sqrt(q_psi)], m)
    D = np.eye(m) - np.eye(m, k=-1)  # (D u)_t = u_t - u_{t-1}; row 0 is u_0, so its target is u_last
    d = np.zeros(m)
    d[0] = u_last
    rows = np.vstack([w[:, None] * G, math.sqrt(r) * D])
    rhs = np.concatenate([-w * h, math.sqrt(r) * d])
    return np.linalg.lstsq(rows, rhs, rcond=None)[0]


# --- racing, and the clean rule -------------------------------------------------------------------
def line_error(result, offsets):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def cap_for(track_name):
    return 900.0 if track_name == "monza-like" else 450.0


def variant(agent, name, track_name, control=None, **constants):
    # A reference with the track's measured grip and line (test-circuit: as shipped), and any constants.
    if track_name != "test-circuit":
        constants["MU_G"] = MEASURED_MU_G[track_name]
        constants["LINE_CONTROL"] = tuple(float(value) for value in control)
    if not constants:
        return agent
    return write_variant(agent, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def race(source, track_name, seed=0):
    return run_race(source, track_name, car=carla_like(noise=True), seed=seed, max_seconds=cap_for(track_name))


def seeds_race(source, track_name):
    return repeat_races(source, track_name, n=len(SEEDS), seeds=SEEDS, car_factory=lambda: carla_like(noise=True),
                        max_seconds=cap_for(track_name))


with shim_installed():
    Unit10, unit10_module_name = load_submission(UNIT_10)
shipped = sys.modules[unit10_module_name]  # the shipped file's functions and constants
track = load_track("test-circuit")
reference = Unit10(track.waypoints, None)  # built once: its line, curvature and segment lengths
offsets = np.array(reference.offsets)  # test-circuit's line as shipped

## 2. U10.1: the stacked solve, and the exact reference on one corner

The horizon stacked one tick at a time against the closed form and the shipped `stack_horizon`, for the three horizons the budget policy can use (40, 20 and 10 ticks), on a made-up speed and curvature run; then the least-squares plan against the shipped `least_squares_plan`, and `x = G u + h` against the model rolled forward one tick at a time.

In [ ]:
for m in (40, 20, 10):
    a = 0.05 * np.linspace(10.0, 40.0, m)  # a_t = dt * v_t
    kappa = 0.06 * np.sin(np.linspace(0.0, 3.0, m))
    e0, psi0 = 0.8, 0.05
    G_ticks, h_ticks = stack_by_ticks(e0, psi0, a, kappa)
    G, h = stack_horizon(e0, psi0, a, kappa)
    G_shipped, h_shipped = shipped.stack_horizon(e0, psi0, a, kappa)
    assert np.allclose(G, G_ticks, rtol=1e-12, atol=1e-12) and np.allclose(h, h_ticks, rtol=1e-12, atol=1e-12)
    assert np.allclose(G, G_shipped, rtol=1e-12, atol=1e-12) and np.allclose(h, h_shipped, rtol=1e-12, atol=1e-12)
    u = plan_least_squares(G, h, 0.01, shipped.MPC_Q_E, shipped.MPC_Q_PSI, shipped.MPC_R_DU)
    assert np.allclose(u, shipped.least_squares_plan(G, h, 0.01), rtol=1e-9, atol=1e-12)
    x = G @ u + h
    e, psi = e0, psi0
    for t in range(m):
        e, psi = e + a[t] * psi, psi + a[t] * (u[t] - kappa[t])
        assert abs(x[2 * t] - e) <= 1e-12 and abs(x[2 * t + 1] - psi) <= 1e-12
    print(f"horizon {m}: the closed form equals the tick-by-tick build and the shipped stack_horizon; "
          f"first command {u[0]:+.5f} 1/m")

The exact reference (the probe `c1`): the tightest corner of `test-circuit`'s line (|κ| 0.0658 1/m), entered 20 waypoints early at a steady 18 m/s, 0.8 m off the line and 0.05 rad off its heading, with the last command 0.
`scipy.optimize.minimize` (SLSQP, with the full-lock bounds on u) solves the **nonlinear** path-frame model, sin ψ and κ cos ψ / (1 − κe); the least-squares plan solves the linearised one; both plans are scored on the nonlinear model.
The line's curvature along the 2 s is read at the distance travelled, by linear interpolation between waypoints.

In [ ]:
def interpolate(points, xs, values):
    # linear interpolation of values (given at the increasing xs) at each of the points
    out = np.zeros(len(points))
    j = 0
    for k, point in enumerate(points):
        while xs[j + 1] < point:
            j += 1
        out[k] = values[j] + (values[j + 1] - values[j]) * (point - xs[j]) / (xs[j + 1] - xs[j])
    return out


kap = np.array(reference.curvature)
seg = np.array(reference.segment_length)
assert abs(np.max(np.abs(kap)) - 0.0658) <= 5e-5
i0 = int(np.argmax(np.abs(kap))) - 20
dt, n_ticks, v = 0.05, 40, 18.0
s = np.concatenate([[0.0], np.cumsum(seg[i0:i0 + 200])])
kref = interpolate(v * dt * np.arange(1, n_ticks + 1), s[:-1], kap[i0:i0 + 200])
x0, u_last = (0.8, 0.05), 0.0
q_e, q_psi, r = shipped.MPC_Q_E, shipped.MPC_Q_PSI, shipped.MPC_R_DU


def nonlinear_cost(u):
    e, p = x0
    total = 0.0
    for k in range(n_ticks):
        e, p = e + dt * v * math.sin(p), p + dt * v * (u[k] - kref[k] * math.cos(p) / (1.0 - kref[k] * e))
        total += q_e * e * e + q_psi * p * p
    return total + r * float(np.sum(np.diff(np.concatenate([[u_last], u])) ** 2))


started = time.perf_counter()
G, h = stack_horizon(x0[0], x0[1], np.zeros(n_ticks) + dt * v, kref)
u_ls = plan_least_squares(G, h, u_last, q_e, q_psi, r)
ls_ms = (time.perf_counter() - started) * 1000.0
u_max = math.tan(shipped.MAX_STEER) / shipped.WHEELBASE
started = time.perf_counter()
exact = minimize(nonlinear_cost, np.zeros(n_ticks), method="SLSQP", bounds=[(-u_max, u_max)] * n_ticks,
                 options={"maxiter": 500, "ftol": 1e-12})
scipy_ms = (time.perf_counter() - started) * 1000.0
above = 100.0 * (nonlinear_cost(u_ls) / exact.fun - 1.0)
print(f"scipy (nonlinear): cost {exact.fun:.5f}; least squares (linear), scored on the nonlinear model: "
      f"{nonlinear_cost(u_ls):.5f}, {above:.2f}% above")
print(f"max |u_LS - u_scipy| {np.max(np.abs(u_ls - exact.x)):.1e} 1/m")
print(f"on this machine: least squares {ls_ms:.2f} ms, scipy {scipy_ms:.0f} ms")
assert exact.success, exact
assert abs(exact.fun - 3.49971) <= 1e-4 and abs(nonlinear_cost(u_ls) - 3.53258) <= 1e-4
assert abs(above - 0.94) <= 0.01 and nonlinear_cost(u_ls) <= 1.02 * exact.fun  # U10.1: within 2%
assert np.max(np.abs(u_ls - exact.x)) <= 1e-3

## 3. U10.7, race-free: a plan that is not finite

A plan built from a non-finite input (here, every curvature of the line set to NaN, as a bad reading would) is never applied.
The agent applies the **next command of the last finite plan** instead, counts it in `fallbacks`, and never calls pure pursuit; once that plan is used up, it repeats the last command applied.

In [ ]:
quiet = write_variant(UNIT_10, workdir / "no-timing.py", {"MPC_BUDGET_MS": "None"})
with shim_installed():
    Quiet, _ = load_submission(quiet)
agent = Quiet(track.waypoints, None)


def no_pursuit(*args):
    raise AssertionError("pure pursuit is never used in MPC mode")


agent.pursuit = no_pursuit
agent.cross_track_error = 0.5  # m left of the line at the start, so the plan has something to correct
delta = agent.mpc_command(0.0, 15.0)
plan = np.array(agent.plan)
assert agent.fallbacks == 0 and np.all(np.isfinite(plan)) and delta == math.atan(shipped.WHEELBASE * plan[0])
agent.curvature = [math.nan] * len(agent.curvature)
for used in (1, 2):
    delta = agent.mpc_command(0.0, 15.0)
    assert agent.fallbacks == used and agent.mpc_last == plan[used] and agent.mpc_queue == [plan[used]]
    assert math.isfinite(delta) and delta == math.atan(shipped.WHEELBASE * plan[used])
agent.plan_used = len(agent.plan)  # the plan used up: the last command applied, again
last = agent.mpc_last
delta = agent.mpc_command(0.0, 15.0)
assert agent.fallbacks == 3 and agent.mpc_last == last and agent.over_budget == 0 and agent.horizon == 40
print(f"three non-finite plans: fallbacks {agent.fallbacks}, commands {plan[1]:+.2e}, {plan[2]:+.2e}, "
      f"then {last:+.2e} 1/m again; pure pursuit never called")

## 4. U10.2: at the same fraction, no faster

At Unit 9's 0.54 on `test-circuit`, seeds 0-4: MPC against Unit 9's pure pursuit (`agents/unit-09.py` as shipped).
The speed profile, not the steering, sets the lap time, so neither is noise-aware better than the other; what MPC adds is the **tracking margin**, which the next sections spend.

In [ ]:
mpc_054 = seeds_race(variant(UNIT_10, "mpc-0.54", "test-circuit", MU_FRACTION=0.54, **NO_TIMING), "test-circuit")
pursuit_054 = seeds_race(UNIT_09, "test-circuit")
mpc_errors = [line_error(result, offsets) for result in mpc_054.results]
pursuit_errors = [line_error(result, offsets) for result in pursuit_054.results]
print(f"f = 0.54, MPC: median {mpc_054.median:.2f} s (seed 0 {mpc_054.times[0]:.2f} s), "
      f"max line error {max(mpc_errors):.2f} m")
print(f"f = 0.54, pursuit: median {pursuit_054.median:.2f} s, max line error {max(pursuit_errors):.2f} m")
print(f"noise-aware better: MPC {noise_aware_better(mpc_054.times, pursuit_054.times)}, "
      f"pursuit {noise_aware_better(pursuit_054.times, mpc_054.times)}")
assert all(clean(result, offsets) for result in mpc_054.results) and max(mpc_errors) <= 1.3
assert abs(mpc_054.times[0] - 279.65) <= TICKS_2
assert all(clean(result, offsets) for result in pursuit_054.results) and abs(pursuit_054.median - 279.60) <= TICKS_2
assert not noise_aware_better(mpc_054.times, pursuit_054.times)
assert not noise_aware_better(pursuit_054.times, mpc_054.times)

## 5. U10.3 and U10.7: the hook, and the pursuit equivalence

At 0.58 on `test-circuit-seed4`, seed 0 (one step above pursuit's family edge, 0.56): Unit 9's pursuit, MPC, and `agents/unit-10.py` with `STEERING = "pursuit"`, which must race exactly like `agents/unit-09.py`, tick for tick.

In [ ]:
seed4 = load_track("test-circuit-seed4")
seed4_control = solve_line(seed4)
seed4_offsets = spline_basis(len(seed4)) @ seed4_control
pursuit_058 = race(variant(UNIT_09, "seed4-pursuit-0.58", "test-circuit-seed4", seed4_control, MU_FRACTION=0.58),
                   "test-circuit-seed4")
mpc_058 = race(variant(UNIT_10, "seed4-mpc-0.58", "test-circuit-seed4", seed4_control, **NO_TIMING),
               "test-circuit-seed4")
as_pursuit = race(variant(UNIT_10, "seed4-as-pursuit", "test-circuit-seed4", seed4_control, STEERING="pursuit"),
                  "test-circuit-seed4")
print(f"test-circuit-seed4, f = 0.58, seed 0: pursuit clean {clean(pursuit_058, seed4_offsets)} "
      f"({line_error(pursuit_058, seed4_offsets):.2f} m from its line); "
      f"MPC clean {clean(mpc_058, seed4_offsets)}, {mpc_058.total_time:.2f} s "
      f"({line_error(mpc_058, seed4_offsets):.2f} m)")
assert not clean(pursuit_058, seed4_offsets)  # U10.3
assert clean(mpc_058, seed4_offsets) and abs(mpc_058.total_time - 271.10) <= TICKS_2
same = all(np.sum(as_pursuit.telemetry[column] != pursuit_058.telemetry[column]) == 0
           for column in as_pursuit.telemetry.dtype.names)
print(f"unit-10.py with STEERING = 'pursuit': {as_pursuit.total_time:.2f} s, identical to unit-09.py: {same}")
assert same and as_pursuit.total_time == pursuit_058.total_time and as_pursuit.collisions == pursuit_058.collisions

## 6. U10.4 and U10.6: the edge, and the lag

At 0.60 on `test-circuit`, seed 0, MPC is clean; the same race with `MPC_LAG = 0` (a model that forgets the queued command) is not.
At 0.62 on `test-circuit-seed1`, seed 0, MPC is not clean: the family edge is 0.60, so the margin rule freezes 0.58.

In [ ]:
mpc_060 = race(variant(UNIT_10, "mpc-0.60", "test-circuit", MU_FRACTION=0.60, **NO_TIMING), "test-circuit")
no_lag = race(variant(UNIT_10, "mpc-lag0-0.60", "test-circuit", MU_FRACTION=0.60, MPC_LAG=0, **NO_TIMING),
              "test-circuit")
print(f"test-circuit, f = 0.60, seed 0: MPC clean {clean(mpc_060, offsets)}, {mpc_060.total_time:.2f} s; "
      f"MPC_LAG = 0 clean {clean(no_lag, offsets)} ({line_error(no_lag, offsets):.2f} m from its line)")
assert clean(mpc_060, offsets) and abs(mpc_060.total_time - 272.40) <= TICKS_2  # U10.4
assert not clean(no_lag, offsets)  # U10.6

seed1 = load_track("test-circuit-seed1")
seed1_control = solve_line(seed1)
seed1_offsets = spline_basis(len(seed1)) @ seed1_control
edge = race(variant(UNIT_10, "seed1-mpc-0.62", "test-circuit-seed1", seed1_control, MU_FRACTION=0.62, **NO_TIMING),
            "test-circuit-seed1")
print(f"test-circuit-seed1, f = 0.62, seed 0: MPC clean {clean(edge, seed1_offsets)}")
assert not clean(edge, seed1_offsets)  # U10.4

## 7. U10.5 and U10.7: the transfer, and the tick budget

On the unseen `test-circuit-seed2026`, seed 0: the shipped 0.58 with timing off, Unit 9's file at its 0.54, and the shipped file with `MPC_BUDGET_MS = 0.0`.
With a budget of 0 every `mpc()` call is over it: the first call halves the horizon to 20, the second to 10, and it stays at 10 (`MPC_MIN_HORIZON`) for the rest of the race, deterministically.
The agent prints one line each time it shortens its horizon.
The race is recorded through a subclass that keeps the agent and the horizon of every `mpc()` call; nothing else about the race changes.

In [ ]:
seed2026 = load_track("test-circuit-seed2026")
seed2026_control = solve_line(seed2026)
seed2026_offsets = spline_basis(len(seed2026)) @ seed2026_control
transfer = race(variant(UNIT_10, "seed2026-mpc-0.58", "test-circuit-seed2026", seed2026_control, **NO_TIMING),
                "test-circuit-seed2026")
unit_09 = race(variant(UNIT_09, "seed2026-pursuit-0.54", "test-circuit-seed2026", seed2026_control),
               "test-circuit-seed2026")
print(f"test-circuit-seed2026, seed 0: MPC (0.58) clean {clean(transfer, seed2026_offsets)}, "
      f"{transfer.total_time:.2f} s; Unit 9 (0.54) clean {clean(unit_09, seed2026_offsets)}, {unit_09.total_time:.2f} s")
assert clean(transfer, seed2026_offsets) and abs(transfer.total_time - 276.85) <= TICKS_2  # U10.5
assert clean(unit_09, seed2026_offsets) and abs(unit_09.total_time - 282.05) <= TICKS_2
assert noise_aware_better([transfer.total_time], [unit_09.total_time])

with shim_installed():
    AllOver, _ = load_submission(variant(UNIT_10, "seed2026-budget0", "test-circuit-seed2026", seed2026_control,
                                         MPC_BUDGET_MS=0.0))
agents, horizons = [], []


class Recorded(AllOver):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        agents.append(self)

    def mpc(self, yaw, speed):
        horizons.append(self.horizon)
        return super().mpc(yaw, speed)


budget0 = race(Recorded, "test-circuit-seed2026")
(budget0_agent,) = agents
steps = len(budget0.step_ms)
print(f"MPC_BUDGET_MS = 0.0: clean {clean(budget0, seed2026_offsets)}, {budget0.total_time:.2f} s; "
      f"{steps} steps, over_budget {budget0_agent.over_budget}, fallbacks {budget0_agent.fallbacks}, "
      f"horizons {horizons[0]}, {horizons[1]}, then {horizons[2]} for the other {steps - 2} calls")
assert clean(budget0, seed2026_offsets) and abs(budget0.total_time - 277.30) <= TICKS_2
assert steps == np.sum(budget0.telemetry["held"] == 0) == len(horizons)  # one mpc() call per step, warm-up included
assert budget0_agent.over_budget == steps and budget0_agent.fallbacks == 0 and budget0_agent.horizon == 10
assert horizons[:2] == [40, 20] and all(value == 10 for value in horizons[2:])

report = check_submission(UNIT_10)  # the shipped file, as shipped (MPC_BUDGET_MS = 5.0)
print(report)
assert report.ok, report.findings

## 8. U10.8: `monza-like`

At 0.64 on `monza-like`, seed 0, on its own line with its measured grip: MPC and Unit 9's pursuit.
Pursuit finishes, but its max line error is just over the 2 m rule, so it is not clean: a tracking-margin result, not a crash.

In [ ]:
monza = load_track("monza-like")
monza_control = solve_line(monza)
monza_offsets = spline_basis(len(monza)) @ monza_control
monza_mpc = race(variant(UNIT_10, "monza-mpc-0.64", "monza-like", monza_control, MU_FRACTION=0.64, **NO_TIMING),
                 "monza-like")
monza_pursuit = race(variant(UNIT_09, "monza-pursuit-0.64", "monza-like", monza_control, MU_FRACTION=0.64),
                     "monza-like")
print(f"monza-like, f = 0.64, seed 0: MPC clean {clean(monza_mpc, monza_offsets)}, {monza_mpc.total_time:.2f} s, "
      f"{line_error(monza_mpc, monza_offsets):.2f} m from its line")
print(f"pursuit: finished {monza_pursuit.finished}, clean {clean(monza_pursuit, monza_offsets)}, "
      f"{line_error(monza_pursuit, monza_offsets):.2f} m from its line (the rule: {CLEAN_ERROR} m)")
assert clean(monza_mpc, monza_offsets) and abs(monza_mpc.total_time - 468.75) <= TICKS_2
assert not clean(monza_pursuit, monza_offsets)

## Unit 10, pinned

| carla-like car, MPC's own line | pure pursuit (Unit 9) | MPC (Unit 10) |
|---|---|---|
| `test-circuit`, f = 0.54, seeds 0-4 | 279.60 s | 279.65 s (seed 0): neither noise-aware better |
| `test-circuit-seed4`, f = 0.58 | not clean | **271.10 s** |
| `test-circuit-seed2026` (unseen) | 282.05 s (f = 0.54, median of five) | **276.85 s** (f = 0.58) |
| `monza-like`, f = 0.64 | not clean (just over 2 m) | 468.75 s |

| the family edge of `MU_FRACTION` | pure pursuit | MPC |
|---|---|---|
| clean on all five training tracks up to | 0.56 | 0.60 |
| frozen by the margin rule | 0.54 | **0.58** |

Without the lag in its model (`MPC_LAG = 0`) MPC is not clean at 0.60; with every call over budget it races on at a 10-tick horizon, clean.
The submission ships `test-circuit`'s line and grip; on another track both are written in with `write_variant`.